# 18 — Subcatchment LID usage

This notebook shows how to assign EPA SWMM LID controls to subcatchments in `swmmx` 0.0.42. LID controls define the device layers; subcatchment LID usage defines where those devices are placed and how much of each catchment they occupy.

## Public API

```python
m.set.subcatchment.lid_controls.add(subcatchment, lid_control, **options)
m.set.subcatchment.lid_controls.remove(subcatchment, lid_control)
m.set.subcatchment.lid_controls.clear(subcatchment)
m.get.subcatchment.lid_controls(ids=None, format=None)
```

Mandatory inputs for `.add()` are: `subcatchment`, `lid_control`, and one sizing option: `area`, `area_percent`, or `occupies_full_subcatchment=True`.

In [ ]:
from swmmx import swmm

m = swmm(new="SI")
m.add.node.junction("J1", x=0, y=0, invert_elevation=10, max_depth=3)
m.add.node.outfall("OUT1", x=100, y=0, invert_elevation=9, type="FREE")
m.add.time.time_series("Rain1", data=[("2026-01-01 00:00", 0.0)])
m.add.hydrology.rain_gage("RG1", format="INTENSITY", interval="00:05", source_type="TIMESERIES", time_series="Rain1")
m.add.hydrology.subcatchment("S1", rain_gage="RG1", outlet="J1", x=25, y=25, area=1.0)
m.add.hydrology.subcatchment("S2", rain_gage="RG1", outlet="J1", x=50, y=25, area=1.0)

## Create LID controls

The control describes the LID device. All EPA SWMM LID control types are supported: bio-retention cell, rain garden, green roof, infiltration trench, permeable pavement, rain barrel, rooftop disconnection, and vegetative swale.

In [ ]:
m.add.hydrology.lid_control("BR1", type="bio_retention_cell", surface={}, soil={}, storage={})
m.add.hydrology.lid_control("RG_LID", type="rain_garden", surface={}, soil={})

print(m.get.lid_control.type())

## Add LID usage with a percent of subcatchment area

`area_percent` is user-friendly. `swmmx` converts it to the per-unit LID area required by SWMM. SI models convert hectares to square meters; US models convert acres to square feet.

In [ ]:
result = m.set.subcatchment.lid_controls.add("S1", "BR1", area_percent=10)
result

## Inspect LID usage

The getter returns a list of usage dictionaries for one subcatchment, or a dictionary grouped by subcatchment when no single ID is selected. Use `format="df"` for a flat table.

In [ ]:
m.get.subcatchment.lid_controls("S1")

In [ ]:
m.get.subcatchment.lid_controls(format="df")

## Full LID Usage Editor options

Optional inputs: `number`, `width`, `initial_saturation`, `from_impervious_percent`, `from_pervious_percent`, `return_flow_to_pervious`, `report_file`, `drain_to`, and `replace`. `width` is auto-estimated from area when omitted.

In [ ]:
m.set.subcatchment.lid_controls.add(
    "S1",
    "BR1",
    area_percent=15,
    number=2,
    width=4.5,
    initial_saturation=25,
    from_impervious_percent=40,
    from_pervious_percent=10,
    return_flow_to_pervious=True,
    report_file="S1_BR1_lid.txt",
    drain_to="S2",
    replace=True,
)

m.get.subcatchment.lid_controls("S1")

## LID occupies the full subcatchment

Use `occupies_full_subcatchment=True` to mirror the SWMM editor setting. The stored SWMM row still uses per-unit area, so `swmmx` derives the flag when reading it back.

In [ ]:
m.set.subcatchment.lid_controls.add("S2", "RG_LID", occupies_full_subcatchment=True)
m.get.subcatchment.lid_controls("S2")

## Replace all assignments in one call

Calling `m.set.subcatchment.lid_controls(...)` replaces the entire LID group for the selected subcatchment.

In [ ]:
m.set.subcatchment.lid_controls(
    [
        {"lid_control": "BR1", "area_percent": 8},
        {"lid_control": "RG_LID", "area_percent": 12},
    ],
    ids="S1",
)

m.get.subcatchment.lid_controls("S1")

## Remove or clear LID usage

Use `remove()` for one control assignment and `clear()` for all LID assignments on one subcatchment.

In [ ]:
m.set.subcatchment.lid_controls.remove("S1", "BR1")
m.get.subcatchment.lid_controls("S1")

In [ ]:
m.set.subcatchment.lid_controls.clear("S1")
m.get.subcatchment.lid_controls("S1")

## Validation rules

`swmmx` validates references, percentages, duplicate assignments, and LID Group Editor limits. Total LID area, total impervious treatment, and total pervious treatment cannot exceed 100% within a subcatchment.

In [ ]:
# Example validation failures:
# m.set.subcatchment.lid_controls.add("S1", "BR1")  # missing area/area_percent/full-subcatchment sizing
# m.set.subcatchment.lid_controls.add("S1", "BR1", area_percent=101)  # percent out of range
# m.set.subcatchment.lid_controls.add("S1", "BR1", area_percent=10, drain_to="UNKNOWN")  # bad drain target